# Split Generation

Generates and saves splits for CV and test

In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path('/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/PBSG')))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [3]:
import importlib
import scripts.run_random_stratified

# After you make a change to the .py file, run this:
importlib.reload(scripts.run_random_stratified)

<module 'scripts.run_random_stratified' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML_clean/Regressor/scripts/run_random_stratified.py'>

In [4]:
from scripts.run_random_stratified import seed_search_random_stratified_reg

best_seed = seed_search_random_stratified_reg(df, n_seeds=50)

Seed search results (top 10 clean seeds by Tm deviation):
 seed  n_test  missing_cls  missing_stereo  tm_mean_dev  tm_std_dev  tm_dev
   33      62            0               0        0.093       0.004   0.097
   24      62            0               0        0.235       0.759   0.994
   10      62            0               0        0.893       0.948   1.842
   36      62            0               0        1.164       1.178   2.343
   39      62            0               0        0.257       3.037   3.294
   42      62            0               0        0.133       3.413   3.546
    7      62            0               0        1.152       2.649   3.801
   34      62            0               0        0.883       3.088   3.971
    2      62            0               0        0.031       4.083   4.114
   20      62            0               0        1.414       2.715   4.129

Best seed: 33  (n_test=62, tm_dev=0.097)
Stratification summary (regression):
  Dataset size:     407
  U

/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
